In [1]:
import json
import duckdb
import pandas as pd
from pathlib import Path

# ==========================================
# SETUP DIRECTORY PATHS & DUCKDB CONNECTION
# ==========================================
BASE_DIR = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
DATA_DIR = BASE_DIR / "data_sets"
RESULT_DIR = BASE_DIR / "results"
ANALYSIS_DIR = BASE_DIR / "analysis"

RESULT_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = ANALYSIS_DIR / "step02_03_ETL_cleaning.duckdb"
con = duckdb.connect(str(DB_PATH))

def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()

FINDINGS = {}

input_delay_file = RESULT_DIR / "combined_delay_imputed.csv"
print(f"Loading imputed delay table from: {input_delay_file.as_posix()} ...")

con.execute(f"""
CREATE OR REPLACE TABLE delay_imputed AS
SELECT 
    CAST(date AS DATE) AS date,
    CAST(train_no AS INTEGER) AS train_no,
    CAST(station_no AS INTEGER) AS station_no,
    CAST(station_name AS VARCHAR) AS station_name,
    CAST(delay AS DOUBLE) AS delay,
    CAST(is_imputed AS BOOLEAN) AS is_imputed
FROM read_csv_auto('{input_delay_file.as_posix()}', header=True);
""")

init_summary = q("""
SELECT 
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE delay IS NULL) AS null_delays,
    COUNT(*) FILTER (WHERE is_imputed = TRUE) AS imputed_rows,
    COUNT(*) FILTER (WHERE is_imputed = FALSE) AS observed_rows
FROM delay_imputed;
""")

print("=== INITIAL DATASET SUMMARY (FROM STEP 2.2) ===")
print(init_summary.to_string(index=False))

FINDINGS["initial_summary"] = init_summary.iloc[0].to_dict()

Loading imputed delay table from: D:/M_Tech_CSE_AI/SEM01/DWDM/project/indian_railway_delay/results/combined_delay_imputed.csv ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== INITIAL DATASET SUMMARY (FROM STEP 2.2) ===
 total_rows  null_delays  imputed_rows  observed_rows
   36629806            0       1032906       35596900


In [2]:
# ==========================================
# STEP 1: NEGATIVE DELAY (EARLY ARRIVAL) VALIDATION
# ==========================================
print("\n=== STEP 1: NEGATIVE DELAY (EARLY ARRIVAL) METRICS ===")

early_arrival_query = """
SELECT 
    COUNT(*) AS total_early_records,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM delay_imputed), 4) AS pct_of_total,
    COUNT(*) FILTER (WHERE is_imputed = FALSE) AS observed_early_records,
    COUNT(*) FILTER (WHERE is_imputed = TRUE) AS imputed_early_records,
    MIN(delay) AS max_early_minutes,
    MAX(delay) AS min_early_minutes,
    ROUND(AVG(delay), 4) AS mean_early_minutes,
    ROUND(MEDIAN(delay), 4) AS median_early_minutes,
    ROUND(STDDEV(delay), 4) AS std_dev_early_minutes
FROM delay_imputed
WHERE delay < 0;
"""

early_metrics = q(early_arrival_query)
print(early_metrics.to_string(index=False))

early_sample = q("""
SELECT date, train_no, station_no, station_name, delay, is_imputed
FROM delay_imputed
WHERE delay < 0
ORDER BY delay ASC
LIMIT 5;
""")
print("\nSample of Extreme Early Arrivals:")
print(early_sample.to_string(index=False))

FINDINGS["step1_negative_delay_validation"] = early_metrics.iloc[0].to_dict()


=== STEP 1: NEGATIVE DELAY (EARLY ARRIVAL) METRICS ===
 total_early_records  pct_of_total  observed_early_records  imputed_early_records  max_early_minutes  min_early_minutes  mean_early_minutes  median_early_minutes  std_dev_early_minutes
              609727        1.6646                  584981                  24746             -120.0               -1.0            -13.1006                 -10.0                11.0904

Sample of Extreme Early Arrivals:
      date  train_no  station_no station_name  delay  is_imputed
2025-07-05     12008           4          MAS -120.0       False
2026-02-03     16329          21         MAJN -120.0       False
2025-09-05     63272          24          MKA -119.0       False
2025-12-25     54322          13          BLM -119.0       False
2025-02-20     63537          14          SNT -118.0       False


In [3]:
# ==========================================
# STEP 2: NOISE REMOVAL (24-HOUR / 1,440-MIN CUTOFF)
# ==========================================
print("\n=== STEP 2: PRE-CUTOFF TAIL INSPECTION & NOISE REMOVAL ===")

pre_cutoff_query = """
SELECT 
    COUNT(*) AS total_rows_before,
    MIN(delay) AS min_delay_before,
    MAX(delay) AS max_delay_before,
    ROUND(AVG(delay), 4) AS global_mean_before,
    ROUND(STDDEV(delay), 4) AS global_std_before,
    ROUND(AVG(delay) FILTER (WHERE delay > 0), 4) AS pos_mean_before,
    ROUND(STDDEV(delay) FILTER (WHERE delay > 0), 4) AS pos_std_before,
    COUNT(*) FILTER (WHERE delay > 1440) AS noise_rows_over_1440,
    COUNT(*) FILTER (WHERE delay > 1440 AND is_imputed = FALSE) AS observed_noise_rows,
    COUNT(*) FILTER (WHERE delay > 1440 AND is_imputed = TRUE) AS imputed_noise_rows
FROM delay_imputed;
"""

pre_cutoff_metrics = q(pre_cutoff_query)
print(pre_cutoff_metrics.to_string(index=False))

noise_sample = q("""
SELECT date, train_no, station_no, station_name, delay, is_imputed
FROM delay_imputed
WHERE delay > 1440
ORDER BY delay DESC
LIMIT 10;
""")
print("\nSample of Dropped Noise Records (delay > 1,440 mins):")
print(noise_sample.to_string(index=False))

# Create cleaned table preserving the `is_imputed` flag
con.execute("""
CREATE OR REPLACE TABLE delay_cleaned AS
SELECT 
    date,
    train_no,
    station_no,
    station_name,
    delay,
    is_imputed
FROM delay_imputed
WHERE delay <= 1440;
""")

post_cutoff_count = q("""
SELECT 
    COUNT(*) AS surviving_rows,
    COUNT(*) FILTER (WHERE is_imputed = FALSE) AS surviving_observed_rows,
    COUNT(*) FILTER (WHERE is_imputed = TRUE) AS surviving_imputed_rows
FROM delay_cleaned;
""")
print("\n--- POST-CUTOFF ROW COUNTS ---")
print(post_cutoff_count.to_string(index=False))

FINDINGS["step2_noise_removal"] = {
    **pre_cutoff_metrics.iloc[0].to_dict(),
    **post_cutoff_count.iloc[0].to_dict()
}


=== STEP 2: PRE-CUTOFF TAIL INSPECTION & NOISE REMOVAL ===
 total_rows_before  min_delay_before  max_delay_before  global_mean_before  global_std_before  pos_mean_before  pos_std_before  noise_rows_over_1440  observed_noise_rows  imputed_noise_rows
          36629806            -120.0          525592.0             36.7497           229.8827          43.6007        249.0261                 14982                14582                 400

Sample of Dropped Noise Records (delay > 1,440 mins):
      date  train_no  station_no station_name    delay  is_imputed
2025-12-31     69206           4         PRTN 525592.0       False
2025-12-31     64565          30          SRE 525585.0       False
2025-12-31     17334          21          MRJ 525569.0       False
2025-12-31     18125          79         PURI 525546.0       False
2025-12-31     67225          38          GDR 525545.0       False
2025-12-31     67286          22          RJY 525525.0       False
2025-11-03     11008           7    

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- POST-CUTOFF ROW COUNTS ---
 surviving_rows  surviving_observed_rows  surviving_imputed_rows
       36614824                 35582318                 1032506


In [4]:
# ==========================================
# STEP 3: OUTLIER BOUNDARY DEFINITION (2 SD & 3 SD)
# ==========================================
print("\n=== STEP 3A: GLOBAL CLEANED DISPERSION (ALL ROWS <= 1440) ===")

global_dispersion_query = """
WITH GlobalStats AS (
    SELECT 
        AVG(delay) AS true_mean,
        STDDEV(delay) AS true_std_dev,
        MIN(delay) AS min_delay,
        MAX(delay) AS max_delay
    FROM delay_cleaned
)
SELECT 
    ROUND(true_mean, 6) AS true_mean,
    ROUND(true_std_dev, 6) AS true_std_dev,
    ROUND(true_mean - (2 * true_std_dev), 6) AS left_wall_2sd,
    min_delay AS actual_min_delay,
    ROUND(true_mean + (2 * true_std_dev), 6) AS outlier_threshold_2sd,
    ROUND(true_mean + (3 * true_std_dev), 6) AS outlier_threshold_3sd,
    (SELECT COUNT(*) FROM delay_cleaned WHERE delay > (true_mean + (2 * true_std_dev))) AS high_risk_records_gt_2sd,
    (SELECT COUNT(*) FROM delay_cleaned WHERE delay > (true_mean + (3 * true_std_dev))) AS critical_records_gt_3sd
FROM GlobalStats;
"""
global_disp = q(global_dispersion_query)
print(global_disp.to_string(index=False))

print("\n=== STEP 3B: POSITIVE-TAIL CLEANED DISPERSION (0 < delay <= 1440) ===")

positive_dispersion_query = """
WITH PosStats AS (
    SELECT 
        AVG(delay) AS pos_true_mean,
        STDDEV(delay) AS pos_true_std_dev
    FROM delay_cleaned
    WHERE delay > 0
)
SELECT 
    ROUND(pos_true_mean, 6) AS pos_true_mean,
    ROUND(pos_true_std_dev, 6) AS pos_true_std_dev,
    ROUND(pos_true_mean + (2 * pos_true_std_dev), 6) AS pos_threshold_2sd,
    ROUND(pos_true_mean + (3 * pos_true_std_dev), 6) AS pos_threshold_3sd,
    (SELECT COUNT(*) FROM delay_cleaned WHERE delay > (pos_true_mean + (2 * pos_true_std_dev))) AS pos_high_risk_records_gt_2sd,
    (SELECT COUNT(*) FROM delay_cleaned WHERE delay > (pos_true_mean + (3 * pos_true_std_dev))) AS pos_critical_records_gt_3sd
FROM PosStats;
"""
pos_disp = q(positive_dispersion_query)
print(pos_disp.to_string(index=False))

print("\n=== STEP 3C: OBSERVED-ONLY (NON-IMPUTED) DISPERSION CHECK ===")
observed_dispersion_query = """
SELECT 
    ROUND(AVG(delay), 6) AS obs_global_mean,
    ROUND(STDDEV(delay), 6) AS obs_global_std,
    ROUND(AVG(delay) FILTER (WHERE delay > 0), 6) AS obs_pos_mean,
    ROUND(STDDEV(delay) FILTER (WHERE delay > 0), 6) AS obs_pos_std
FROM delay_cleaned
WHERE is_imputed = FALSE;
"""
obs_disp = q(observed_dispersion_query)
print(obs_disp.to_string(index=False))

FINDINGS["step3_outlier_boundaries"] = {
    "global_cleaned": global_disp.iloc[0].to_dict(),
    "positive_tail_cleaned": pos_disp.iloc[0].to_dict(),
    "observed_only_check": obs_disp.iloc[0].to_dict()
}


=== STEP 3A: GLOBAL CLEANED DISPERSION (ALL ROWS <= 1440) ===
 true_mean  true_std_dev  left_wall_2sd  actual_min_delay  outlier_threshold_2sd  outlier_threshold_3sd  high_risk_records_gt_2sd  critical_records_gt_3sd
  35.59942     66.026817     -96.454213            -120.0             167.653054              233.67987                   1273458                   659943

=== STEP 3B: POSITIVE-TAIL CLEANED DISPERSION (0 < delay <= 1440) ===
 pos_true_mean  pos_true_std_dev  pos_threshold_2sd  pos_threshold_3sd  pos_high_risk_records_gt_2sd  pos_critical_records_gt_3sd
     42.247231          69.61557         181.478371         251.093941                       1094178                       563763

=== STEP 3C: OBSERVED-ONLY (NON-IMPUTED) DISPERSION CHECK ===
 obs_global_mean  obs_global_std  obs_pos_mean  obs_pos_std
       35.677161       65.849781     42.268575     69.37937


In [5]:
# ==========================================
# STEP 4: CONCEPT HIERARCHY GENERATION (TIME DIMENSION)
# ==========================================
print("\n=== STEP 4: GENERATING TIME DIMENSION (dim_date.csv) ===")

# Generate dim_date from the distinct calendar dates present in delay_cleaned
# Also check whether the full date spine between min(date) and max(date) has any missing calendar days
date_check = q("""
SELECT 
    MIN(date) AS min_date,
    MAX(date) AS max_date,
    COUNT(DISTINCT date) AS distinct_dates_in_delay,
    DATEDIFF('day', MIN(date), MAX(date)) + 1 AS expected_calendar_days
FROM delay_cleaned;
""")
print(date_check.to_string(index=False))

con.execute("""
CREATE OR REPLACE TABLE dim_date AS
WITH DateSpine AS (
    SELECT DISTINCT date AS d
    FROM delay_cleaned
)
SELECT 
    STRFTIME(d, '%Y-%m-%d') AS date_id,
    EXTRACT(YEAR FROM d) AS year,
    EXTRACT(QUARTER FROM d) AS quarter,
    EXTRACT(MONTH FROM d) AS month,
    STRFTIME(d, '%B') AS month_name,
    EXTRACT(DAY FROM d) AS day_of_month,
    STRFTIME(d, '%A') AS day_name,
    EXTRACT(ISODOW FROM d) AS day_of_week,
    CASE WHEN EXTRACT(ISODOW FROM d) IN (6, 7) THEN TRUE ELSE FALSE END AS is_weekend
FROM DateSpine
ORDER BY d;
""")

dim_date_sample = q("SELECT * FROM dim_date LIMIT 5;")
print("\nSample of dim_date:")
print(dim_date_sample.to_string(index=False))

dim_date_file = RESULT_DIR / "dim_date.csv"
con.execute(f"COPY dim_date TO '{dim_date_file.as_posix()}' (HEADER, DELIMITER ',');")
print(f"\n-> Saved {q('SELECT COUNT(*) AS n FROM dim_date')['n'][0]} rows to {dim_date_file.as_posix()}")

FINDINGS["step4_dim_date"] = {
    "min_date": str(date_check["min_date"][0]),
    "max_date": str(date_check["max_date"][0]),
    "distinct_dates_in_delay": int(date_check["distinct_dates_in_delay"][0]),
    "expected_calendar_days": int(date_check["expected_calendar_days"][0])
}


=== STEP 4: GENERATING TIME DIMENSION (dim_date.csv) ===
  min_date   max_date  distinct_dates_in_delay  expected_calendar_days
2025-02-08 2026-02-07                      365                     365

Sample of dim_date:
   date_id  year  quarter  month month_name  day_of_month  day_name  day_of_week  is_weekend
2025-02-08  2025        1      2   February             8  Saturday            6        True
2025-02-09  2025        1      2   February             9    Sunday            7        True
2025-02-10  2025        1      2   February            10    Monday            1       False
2025-02-11  2025        1      2   February            11   Tuesday            2       False
2025-02-12  2025        1      2   February            12 Wednesday            3       False

-> Saved 365 rows to D:/M_Tech_CSE_AI/SEM01/DWDM/project/indian_railway_delay/results/dim_date.csv


In [6]:
# ==========================================
# STEP 5: FINAL VERIFICATION & EXPORT
# ==========================================
print("\n=== STEP 5: FINAL VERIFICATION & EXPORT ===")

final_audit = q("""
SELECT 
    COUNT(*) AS total_cleaned_rows,
    COUNT(*) FILTER (WHERE delay IS NULL) AS null_delays,
    COUNT(*) FILTER (WHERE is_imputed IS NULL) AS null_imputation_flags,
    COUNT(*) FILTER (WHERE is_imputed = TRUE) AS total_imputed_rows,
    COUNT(*) FILTER (WHERE is_imputed = FALSE) AS total_observed_rows,
    ROUND(COUNT(*) FILTER (WHERE is_imputed = TRUE) * 100.0 / COUNT(*), 4) AS pct_imputed,
    MIN(delay) AS final_min_delay,
    MAX(delay) AS final_max_delay
FROM delay_cleaned;
""")
print(final_audit.to_string(index=False))

FINDINGS["final_verification"] = final_audit.iloc[0].to_dict()

# Export cleaned delay table (with is_imputed flag intact)
cleaned_delay_file = RESULT_DIR / "combined_delay_cleaned.csv"
print(f"\nExporting delay_cleaned to {cleaned_delay_file.as_posix()} ...")
con.execute(f"COPY delay_cleaned TO '{cleaned_delay_file.as_posix()}' (HEADER, DELIMITER ',');")
print("-> Export complete.")

# Drop staging table to keep the .duckdb file compact
con.execute("DROP TABLE IF EXISTS delay_imputed;")

# Save findings JSON
findings_file = RESULT_DIR / "step02_03_ETL_FINDINGS.json"
with open(findings_file, "w") as f:
    json.dump(FINDINGS, f, indent=4)
print(f"-> Saved findings to {findings_file.as_posix()}")

con.close()


=== STEP 5: FINAL VERIFICATION & EXPORT ===
 total_cleaned_rows  null_delays  null_imputation_flags  total_imputed_rows  total_observed_rows  pct_imputed  final_min_delay  final_max_delay
           36614824            0                      0             1032506             35582318       2.8199           -120.0           1440.0

Exporting delay_cleaned to D:/M_Tech_CSE_AI/SEM01/DWDM/project/indian_railway_delay/results/combined_delay_cleaned.csv ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

-> Export complete.
-> Saved findings to D:/M_Tech_CSE_AI/SEM01/DWDM/project/indian_railway_delay/results/step02_03_ETL_FINDINGS.json
